# Deteccao de Objetos com YOLOS-Small

Este notebook usa o modelo `hustvl/yolos-small` do HuggingFace para detectar objetos nas imagens da pasta `images`.

O YOLOS (You Only Look at One Sequence) e um detector de objetos baseado em Vision Transformer, treinado no dataset COCO (91 classes).

## Padrao de documentacao deste notebook
- Toda nova etapa deve ter **titulo** e **descricao** em celula markdown.
- Toda celula de codigo deve incluir **comentarios curtos** explicando a intencao.

## 1) Imports e configuracao inicial

Importa as bibliotecas necessarias: `transformers` para o modelo e processador, `PIL` para manipulacao de imagens, `matplotlib` para visualizacao e `torch` para inferencia.

In [ ]:
# Importa bibliotecas basicas para sistema de arquivos.
from pathlib import Path
from pprint import pprint

# Importa PyTorch para inferencia.
import torch

# Importa PIL para carregar imagens.
from PIL import Image, ImageDraw, ImageFont

# Importa classes do HuggingFace transformers para deteccao de objetos.
from transformers import AutoImageProcessor, AutoModelForObjectDetection

# Importa matplotlib para exibicao dos resultados visuais.
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Define caminho da pasta de imagens e dispositivo de execucao.
images_dir = Path("images")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"torch: {torch.__version__}")
print(f"device: {device}")

## 2) Carregamento do modelo e processador de imagem

Baixa e inicializa o modelo YOLOS-Small pre-treinado no COCO e seu processador de imagem correspondente.

In [ ]:
# Define identificador do modelo no HuggingFace Hub.
model_name = "hustvl/yolos-small"

# Carrega o processador de imagem (resize, normalizacao, etc.).
processor = AutoImageProcessor.from_pretrained(model_name)

# Carrega o modelo de deteccao de objetos com pesos pre-treinados.
model = AutoModelForObjectDetection.from_pretrained(model_name)
model.to(device)
model.eval()

# Exibe o mapeamento de classes COCO disponivel no modelo.
id2label = model.config.id2label
print(f"Modelo carregado: {model_name}")
print(f"Numero de classes COCO: {len(id2label)}")
print(f"Exemplos: {dict(list(id2label.items())[:5])}")

## 3) Deteccao de objetos em todas as imagens

Percorre cada imagem da pasta `images`, roda inferencia com o YOLOS e filtra deteccoes acima de um limiar de confianca.

In [ ]:
# Limiar minimo de confianca para aceitar uma deteccao.
CONFIDENCE_THRESHOLD = 0.7

# Extensoes de imagem suportadas.
valid_ext = {".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp"}

# Lista arquivos validos na pasta de entrada.
image_paths = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in valid_ext])
if not image_paths:
    raise FileNotFoundError("Nenhuma imagem encontrada na pasta 'images'.")

# Acumula resultados estruturados de deteccao por imagem.
all_results = []

with torch.inference_mode():
    for image_path in image_paths:
        # Abre a imagem em RGB.
        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        # Pre-processa a imagem no formato esperado pelo modelo.
        inputs = processor(images=image, return_tensors="pt").to(device)

        # Executa inferencia e obtem logits e bounding boxes.
        outputs = model(**inputs)

        # Converte saidas do modelo em deteccoes com coordenadas absolutas.
        target_sizes = torch.tensor([[height, width]], device=device)
        results = processor.post_process_object_detection(
            outputs, threshold=CONFIDENCE_THRESHOLD, target_sizes=target_sizes
        )[0]

        # Monta lista de deteccoes filtradas para esta imagem.
        detections = []
        for score, label_id, box in zip(results["scores"], results["labels"], results["boxes"]):
            x_min, y_min, x_max, y_max = box.cpu().tolist()
            detections.append({
                "label": id2label[label_id.item()],
                "confidence": round(score.item(), 4),
                "bbox": [round(v, 1) for v in [x_min, y_min, x_max, y_max]],
            })

        all_results.append({
            "image": image_path.name,
            "image_path": str(image_path),
            "num_detections": len(detections),
            "detections": detections,
        })

        print(f"{image_path.name}: {len(detections)} objeto(s) detectado(s)")

print(f"\nTotal de imagens processadas: {len(all_results)}")

## 4) Resultados estruturados

Exibe os resultados completos de deteccao (classe, confianca, bounding box) para cada imagem.

In [ ]:
# Exibe detalhes estruturados de cada deteccao por imagem.
pprint(all_results, sort_dicts=False)

## 5) Visualizacao com bounding boxes

Para cada imagem, exibe a foto original com retangulos coloridos sobre os objetos detectados, junto com o nome da classe e a confianca.

In [ ]:
# Paleta de cores para diferenciar classes distintas visualmente.
COLORS = [
    "#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7",
    "#DDA0DD", "#98D8C8", "#F7DC6F", "#82E0AA", "#F1948A",
]

for item in all_results:
    # Carrega a imagem original para exibicao.
    img = Image.open(item["image_path"]).convert("RGB")

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img)

    # Mapeia cada classe unica a uma cor fixa para consistencia visual.
    unique_labels = list({d["label"] for d in item["detections"]})
    label_color_map = {label: COLORS[i % len(COLORS)] for i, label in enumerate(sorted(unique_labels))}

    # Desenha bounding boxes e rotulos sobre a imagem.
    for det in item["detections"]:
        x_min, y_min, x_max, y_max = det["bbox"]
        w = x_max - x_min
        h = y_max - y_min
        color = label_color_map[det["label"]]

        # Retangulo da bounding box.
        rect = patches.Rectangle(
            (x_min, y_min), w, h,
            linewidth=2.5, edgecolor=color, facecolor="none",
        )
        ax.add_patch(rect)

        # Texto com classe e confianca acima da bounding box.
        label_text = f"{det['label']} {det['confidence']:.0%}"
        ax.text(
            x_min, y_min - 6, label_text,
            fontsize=11, fontweight="bold", color="white",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.85),
        )

    ax.set_title(f"{item['image']} — {item['num_detections']} deteccao(oes)", fontsize=13, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()